In [1]:
!python comparador_precios.py --sitios cimexcolombiasas.com,biosmedic.com,saluti.com.co

python3: can't open file '/content/comparador_precios.py': [Errno 2] No such file or directory


In [2]:
#!/usr/bin/env python3
"""
NEXOR - Comparador de precios GMD (v2)

Estrategia (para no saturar los sitios ni recibir HTTP 429):
  1. Descarga el CATALOGO de cada tienda en pocas peticiones:
       a) Shopify      -> /products.json          (250 productos por peticion)
       b) WooCommerce  -> /wp-json/wc/store/v1/products (100 por peticion)
       c) Respaldo     -> sitemap + JSON-LD de las paginas de producto (limitado)
  2. Guarda todo en SQLite (Datos/comparador.db).
  3. Cruza LOCALMENTE los SKU contra el catalogo descargado.
  4. Exporta Excel con 6 hojas y conserva historial entre ejecuciones.

Reglas de respeto:
  - Obedece robots.txt. Si el sitio lo restringe -> ACCESO_RESTRINGIDO, no se consulta.
  - Pausa entre peticiones (por dominio) y respeta Retry-After.
  - Ante HTTP 429 prolongado detiene ese dominio y lo registra (LIMITADO_429).
  - No accede a portales con login ni intenta evadir restricciones.

Uso:
  python comparador_precios.py                      # todo
  python comparador_precios.py --sitios saluti.com.co,medicolsa.com.co
  python comparador_precios.py --solo-cruce         # reusa catalogos ya descargados
  python comparador_precios.py --sin-fallback       # solo Shopify/WooCommerce
"""
import argparse
import html
import json
import logging
import random
import re
import sqlite3
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ----------------------------------------------------------------- configuracion
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:  # Colab/Jupyter: usa la carpeta de trabajo actual
    BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Datos"
REPORTS_DIR = BASE_DIR / "Reports"
LOGS_DIR = BASE_DIR / "Logs"
DB_PATH = DATA_DIR / "comparador.db"

UA = "NEXOR-PriceBot/2.0 (comparador interno de precios; contacto: ventas@nexor.co)"
TIMEOUT = 30
PAUSA = 3.0              # segundos entre peticiones al mismo dominio
MAX_ESPERA_429 = 120     # si Retry-After supera esto, se detiene el dominio
MAX_PAGINAS_API = 200    # tope de paginas de catalogo por sitio
MAX_PAGINAS_FALLBACK = 150

TIENDAS = [
    "https://tiendamedicalospinos.com", "https://tmvco.com.co", "https://saluti.com.co",
    "https://ortopedicosfuturo.com", "https://gmd.com.co", "https://allers.com.co",
    "https://tecnomedica.com.co", "https://medicolsa.com.co", "https://novamedicasas.com",
    "https://imecolombia.com", "https://coimpormedica.com", "https://distrimedics.com",
    "https://promedco.com", "https://cimexcolombiasas.com", "https://tiendamedica.com.co",
    "https://tiendasmedicascolombia.com", "https://caldasmedicas.com",
    "https://corazonmedico.com.co", "https://dismecol.com", "https://tmmedicas.com",
    "https://labotekcolombia.com", "https://sidsalud.com", "https://medicox.com.co",
    "https://intercomercialmedica.com", "https://ortopedicos.com.co",
    "https://morthopedics.com", "https://biomaxcolombia.com", "https://quirurgicosgl.com",
    "https://medical-colombia.com", "https://bbraun.com", "https://hospitalaria.com.co",
    "https://sumimedical.com", "https://disortho.com.co", "https://furesa.co",
    "https://almar-medica.com", "https://biosmedic.com",
]

# Sitios que NO se deben consultar automaticamente (portal de distribuidores con login)
EXCLUIDOS = {"novamedicasas.com": "Precios solo en portal de distribuidores autorizados"}

log = logging.getLogger("nexor")


# ----------------------------------------------------------------- utilidades
def norm(s):
    return re.sub(r"[^A-Z0-9]", "", str(s or "").upper())


def to_float(v, api=False):
    if v is None or v == "":
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if api:
        try:
            return float(v)
        except ValueError:
            pass
    s = re.sub(r"[^\d.,]", "", str(v))
    if not s:
        return None
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", s):      # 1.234.567,89
        s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", s):    # 1,234,567.89
        s = s.replace(",", "")
    elif "," in s and "." not in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def leer_skus(ruta):
    skus, vistos = [], set()
    for linea in Path(ruta).read_text(encoding="utf-8", errors="ignore").splitlines():
        s = linea.strip()
        if not s or s.upper() == "N/D" or " " in s or "…" in s:
            continue
        if s not in vistos:
            vistos.add(s)
            skus.append(s)
    return skus


# ----------------------------------------------------------------- cliente HTTP respetuoso
class Cliente:
    def __init__(self, pausa=PAUSA):
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": UA, "Accept-Language": "es-CO,es;q=0.9"})
        self.pausa = pausa
        self.robots = {}
        self.bloqueado = {}   # base -> estado
        self.ultimo = {}

    def robots_de(self, base):
        if base in self.robots:
            return self.robots[base]
        rp = RobotFileParser()
        try:
            r = self.s.get(base + "/robots.txt", timeout=TIMEOUT)
            if r.status_code == 200:
                rp.parse(r.text.splitlines())
            elif 400 <= r.status_code < 500:
                rp.parse([])
            else:
                rp.parse(["User-agent: *", "Disallow: /"])
        except requests.RequestException:
            rp.parse([])
        self.robots[base] = rp
        return rp

    def _esperar(self, base):
        falta = self.pausa + random.uniform(0, 1.0) - (time.time() - self.ultimo.get(base, 0))
        if falta > 0:
            time.sleep(falta)
        self.ultimo[base] = time.time()

    def get(self, base, url, params=None):
        """Devuelve (respuesta|None, motivo|None)."""
        if base in self.bloqueado:
            return None, self.bloqueado[base]
        if not self.robots_de(base).can_fetch(UA, url):
            return None, "ROBOTS"
        for intento in range(3):
            self._esperar(base)
            try:
                r = self.s.get(url, params=params, timeout=TIMEOUT)
            except requests.RequestException as e:
                log.warning("%s: error de red %s", url, e)
                return None, "ERROR_RED"
            if r.status_code == 429:
                ra = r.headers.get("Retry-After", "")
                espera = int(ra) if ra.isdigit() else 30 * (intento + 1)
                log.warning("%s: HTTP 429 (espera %ss)", base, espera)
                if espera > MAX_ESPERA_429 or intento == 2:
                    self.bloqueado[base] = "LIMITADO_429"
                    log.error("%s: dominio detenido por HTTP 429", base)
                    return None, "LIMITADO_429"
                time.sleep(espera)
                continue
            if r.status_code >= 400:
                return None, f"HTTP_{r.status_code}"
            return r, None
        return None, "LIMITADO_429"


# ----------------------------------------------------------------- recolectores de catalogo
def catalogo_shopify(c, base):
    filas, pagina = [], 1
    while pagina <= MAX_PAGINAS_API:
        r, motivo = c.get(base, f"{base}/products.json", params={"limit": 250, "page": pagina})
        if r is None:
            return (filas or None), motivo
        try:
            prods = r.json().get("products")
        except (ValueError, AttributeError):
            return None, "NO_API"
        if prods is None:
            return None, "NO_API"
        if not prods:
            break
        for p in prods:
            for v in p.get("variants", []):
                var = v.get("title")
                titulo = p.get("title", "")
                if var and var != "Default Title":
                    titulo += f" - {var}"
                filas.append(dict(
                    sku=(v.get("sku") or "").strip(), titulo=titulo, marca=p.get("vendor", ""),
                    precio=to_float(v.get("price"), api=True),
                    precio_anterior=to_float(v.get("compare_at_price"), api=True),
                    disponibilidad="Disponible" if v.get("available") else "Agotado",
                    url=f"{base}/products/{p.get('handle')}", fuente="shopify"))
        pagina += 1
    return filas, None


def catalogo_woo(c, base):
    filas, pagina = [], 1
    while pagina <= MAX_PAGINAS_API:
        r, motivo = c.get(base, f"{base}/wp-json/wc/store/v1/products",
                          params={"per_page": 100, "page": pagina})
        if r is None:
            return (filas or None), motivo
        try:
            items = r.json()
        except ValueError:
            return None, "NO_API"
        if not isinstance(items, list):
            return None, "NO_API"
        if not items:
            break
        for it in items:
            pr = it.get("prices") or {}
            dec = 10 ** int(pr.get("currency_minor_unit", 0) or 0)
            precio = to_float(pr.get("price"), api=True)
            previo = to_float(pr.get("regular_price"), api=True)
            marcas = it.get("brands") or []
            filas.append(dict(
                sku=(it.get("sku") or "").strip(), titulo=html.unescape(it.get("name", "")),
                marca=", ".join(m.get("name", "") for m in marcas if isinstance(m, dict)),
                precio=precio / dec if precio is not None else None,
                precio_anterior=(previo / dec if previo is not None and previo != precio else None),
                disponibilidad="Disponible" if it.get("is_in_stock") else "Agotado",
                url=it.get("permalink", ""), fuente="woocommerce"))
        if len(items) < 100:
            break
        pagina += 1
    return filas, None


def _sitemaps(c, base, profundidad=0, urls_xml=None):
    if urls_xml is None:
        urls_xml = list(c.robots_de(base).site_maps() or []) or [
            f"{base}/sitemap.xml", f"{base}/sitemap_index.xml", f"{base}/wp-sitemap.xml"]
    paginas = []
    urls_xml = sorted(urls_xml, key=lambda u: 0 if "product" in u.lower() else 1)[:40]
    for u in urls_xml:
        r, _ = c.get(base, u)
        if r is None:
            continue
        locs = re.findall(r"<loc>\s*(.*?)\s*</loc>", r.text, flags=re.S)
        anidados = [x for x in locs if x.lower().endswith(".xml")]
        if anidados and profundidad < 2:
            paginas += _sitemaps(c, base, profundidad + 1, anidados)
        paginas += [x for x in locs if not x.lower().endswith(".xml")]
        if paginas:
            break
    return paginas


def _jsonld_producto(soup):
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except (ValueError, TypeError):
            continue
        pila = data if isinstance(data, list) else [data]
        while pila:
            d = pila.pop()
            if isinstance(d, dict):
                if "@graph" in d:
                    pila += d["@graph"]
                tipo = d.get("@type")
                tipos = tipo if isinstance(tipo, list) else [tipo]
                if "Product" in tipos:
                    return d
    return None


def catalogo_fallback(c, base, skus, max_paginas):
    todas = _sitemaps(c, base)
    patron = re.compile(r"/(product|producto|productos|products|shop|tienda|item)s?/", re.I)
    prod = [u for u in dict.fromkeys(todas) if patron.search(urlparse(u).path)]
    if not prod:
        return None, "SIN_SITEMAP_PRODUCTOS"
    claves = [s.lower() for s in skus if len(s) >= 5]
    prod.sort(key=lambda u: 0 if any(k in u.lower() for k in claves) else 1)
    filas = []
    for u in prod[:max_paginas]:
        r, motivo = c.get(base, u)
        if r is None:
            if motivo == "LIMITADO_429":
                break
            continue
        p = _jsonld_producto(BeautifulSoup(r.text, "html.parser"))
        if not p:
            continue
        of = p.get("offers") or {}
        of = of[0] if isinstance(of, list) and of else of
        of = of if isinstance(of, dict) else {}
        disp = str(of.get("availability", ""))
        marca = p.get("brand")
        marca = marca.get("name", "") if isinstance(marca, dict) else (marca or "")
        filas.append(dict(
            sku=str(p.get("sku") or p.get("mpn") or "").strip(),
            titulo=html.unescape(str(p.get("name", ""))), marca=marca,
            precio=to_float(of.get("price") or of.get("lowPrice")), precio_anterior=None,
            disponibilidad=("Disponible" if "InStock" in disp else "Agotado" if "OutOfStock" in disp
                            else "Sin informacion"),
            url=u, fuente="jsonld"))
    return (filas or None), None


def obtener_catalogo(c, base, skus, args):
    dom = urlparse(base).netloc.replace("www.", "")
    if dom in EXCLUIDOS:
        return [], "ACCESO_RESTRINGIDO", "-", EXCLUIDOS[dom]
    if not c.robots_de(base).can_fetch(UA, base + "/"):
        return [], "ACCESO_RESTRINGIDO", "-", "robots.txt no permite el acceso automatizado"
    motivos = []
    for nombre, fn in (("shopify", catalogo_shopify), ("woocommerce", catalogo_woo)):
        filas, motivo = fn(c, base)
        if filas:
            return filas, "OK", nombre, ""
        motivos.append(motivo)
        if base in c.bloqueado:
            return [], "LIMITADO_429", nombre, "El servidor limito las solicitudes (HTTP 429)"
    if not args.sin_fallback:
        filas, motivo = catalogo_fallback(c, base, skus, args.max_paginas)
        if filas:
            return filas, "OK_PARCIAL", "jsonld", f"Cobertura parcial (max {args.max_paginas} paginas)"
        motivos.append(motivo)
        if base in c.bloqueado:
            return [], "LIMITADO_429", "jsonld", "El servidor limito las solicitudes (HTTP 429)"
    if motivos and all(m == "ROBOTS" for m in motivos if m):
        return [], "ACCESO_RESTRINGIDO", "-", "robots.txt"
    return [], "CATALOGO_NO_DETECTADO", "-", "Requiere adaptador especifico: " + ", ".join(m for m in motivos if m)


# ----------------------------------------------------------------- base de datos
def init_db():
    DATA_DIR.mkdir(exist_ok=True)
    con = sqlite3.connect(DB_PATH)
    con.executescript("""
    CREATE TABLE IF NOT EXISTS catalogo(dominio TEXT, sku TEXT, titulo TEXT, marca TEXT, precio REAL,
        precio_anterior REAL, disponibilidad TEXT, url TEXT, fuente TEXT, fecha TEXT);
    CREATE TABLE IF NOT EXISTS estados(dominio TEXT PRIMARY KEY, estado TEXT, metodo TEXT,
        productos INTEGER, nota TEXT, fecha TEXT);
    CREATE TABLE IF NOT EXISTS historial(ejecucion TEXT, sku TEXT, competidor TEXT, producto TEXT,
        marca TEXT, precio REAL, precio_anterior REAL, disponibilidad TEXT, url TEXT, coincidencia TEXT);
    """)
    return con


def guardar_catalogo(con, dom, filas, estado, metodo, nota, ahora):
    con.execute("DELETE FROM catalogo WHERE dominio=?", (dom,))
    con.executemany(
        "INSERT INTO catalogo VALUES(?,?,?,?,?,?,?,?,?,?)",
        [(dom, f["sku"], f["titulo"], f["marca"], f["precio"], f["precio_anterior"],
          f["disponibilidad"], f["url"], f["fuente"], ahora) for f in filas])
    con.execute("INSERT OR REPLACE INTO estados VALUES(?,?,?,?,?,?)",
                (dom, estado, metodo, len(filas), nota, ahora))
    con.commit()


# ----------------------------------------------------------------- cruce de SKU
def cruzar(skus, cat):
    cat = cat.reset_index(drop=True)
    cat["sku_norm"] = cat["sku"].fillna("").map(norm)
    textos = (cat["sku"].fillna("") + " | " + cat["titulo"].fillna("") + " | "
              + cat["url"].fillna("")).str.upper().tolist()
    idx = cat.groupby("sku_norm").indices
    filas = []

    def reg(sku, i, tipo):
        r = cat.iloc[i]
        obs = ("Verificar que presentacion/unidad de venta sea la misma" if tipo == "EXACTA"
               else "El SKU aparece en titulo/URL, no en el campo SKU: validar manualmente")
        filas.append(dict(SKU=sku, Competidor=r["dominio"], Producto=r["titulo"], Marca=r["marca"],
                          Precio=r["precio"], Precio_anterior=r["precio_anterior"],
                          Disponibilidad=r["disponibilidad"], URL=r["url"], Coincidencia=tipo,
                          Fecha_catalogo=r["fecha"], Observaciones=obs))

    for sku in skus:
        n = norm(sku)
        exactas = set(idx[n]) if len(n) >= 3 and n in idx else set()
        for i in exactas:
            reg(sku, i, "EXACTA")
        if len(n) >= 5:
            su = sku.upper()
            pat = re.compile(rf"(?<![A-Z0-9]){re.escape(su)}(?![A-Z0-9])")
            for i, t in enumerate(textos):
                if i not in exactas and su in t and pat.search(t):
                    reg(sku, i, "PROBABLE")
    cols = ["SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior", "Disponibilidad",
            "URL", "Coincidencia", "Fecha_catalogo", "Observaciones"]
    return pd.DataFrame(filas, columns=cols)


# ----------------------------------------------------------------- Excel
def exportar(res, skus, estados, hist, ruta):
    ex = res[res["Coincidencia"] == "EXACTA"].dropna(subset=["Precio"])
    if len(ex):
        g = ex.groupby("SKU")["Precio"]
        mejor = pd.DataFrame({"Precio_minimo": g.min(), "Precio_maximo": g.max(),
                              "Precio_promedio": g.mean().round(0), "Competidores": g.count()})
        imin = ex.loc[g.idxmin()].set_index("SKU")
        mejor["Competidor_mas_barato"] = imin["Competidor"]
        mejor["URL"] = imin["URL"]
        mejor = mejor.reset_index()
    else:
        mejor = pd.DataFrame(columns=["SKU", "Precio_minimo", "Precio_maximo", "Precio_promedio",
                                      "Competidores", "Competidor_mas_barato", "URL"])
    con_algo = set(res["SKU"])
    sin = pd.DataFrame({"SKU": [s for s in skus if s not in con_algo]})
    sin["Nota"] = "Sin coincidencia en los catalogos consultados (no implica que no exista en el mercado)"
    ok = estados[estados["estado"].isin(["OK", "OK_PARCIAL"])].shape[0]
    resumen = pd.DataFrame({"Indicador": [
        "SKU unicos consultados", "Sitios configurados", "Sitios con catalogo obtenido",
        "Sitios restringidos / limitados / sin catalogo", "SKU con coincidencia EXACTA",
        "SKU solo con coincidencia PROBABLE", "SKU sin coincidencia", "Fecha de ejecucion"],
        "Valor": [len(skus), len(estados), ok, len(estados) - ok,
                  res.loc[res["Coincidencia"] == "EXACTA", "SKU"].nunique(),
                  len(set(res.loc[res["Coincidencia"] == "PROBABLE", "SKU"])
                      - set(res.loc[res["Coincidencia"] == "EXACTA", "SKU"])),
                  len(sin), datetime.now().strftime("%Y-%m-%d %H:%M")]})
    estados = estados.rename(columns={"dominio": "Sitio", "estado": "Estado", "metodo": "Metodo",
                                      "productos": "Productos_en_catalogo", "nota": "Nota",
                                      "fecha": "Fecha"})
    with pd.ExcelWriter(ruta, engine="openpyxl") as w:
        for nombre, df in (("Resumen", resumen), ("Comparacion", res), ("Mejor precio", mejor),
                           ("Sin coincidencias", sin), ("Estado sitios", estados), ("Historial", hist)):
            df.to_excel(w, sheet_name=nombre, index=False)
            ws = w.sheets[nombre]
            ws.freeze_panes = "A2"
            for col in ws.columns:
                ancho = max(len(str(col[0].value or "")), *(len(str(x.value or "")) for x in col[:200]))
                ws.column_dimensions[col[0].column_letter].width = min(max(ancho + 2, 10), 60)
                if str(col[0].value).lower().startswith(("precio", "mejor")):
                    for x in col[1:]:
                        x.number_format = '"$"#,##0'
    log.info("Excel generado: %s", ruta)


# ----------------------------------------------------------------- principal
def main():
    ap = argparse.ArgumentParser(description="NEXOR - Comparador de precios GMD")
    ap.add_argument("--skus", default=str(BASE_DIR / "sku_gmd.txt"))
    ap.add_argument("--sitios", default="", help="dominios separados por coma (opcional)")
    ap.add_argument("--solo-cruce", action="store_true", help="no descarga; usa catalogos guardados")
    ap.add_argument("--sin-fallback", action="store_true", help="omite el respaldo sitemap+JSON-LD")
    ap.add_argument("--max-paginas", type=int, default=MAX_PAGINAS_FALLBACK)
    ap.add_argument("--pausa", type=float, default=PAUSA)
    args, _ = ap.parse_known_args()

    for d in (DATA_DIR, REPORTS_DIR, LOGS_DIR):
        d.mkdir(exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                        handlers=[logging.FileHandler(LOGS_DIR / "ejecucion.log", encoding="utf-8"),
                                  logging.StreamHandler()])
    skus = leer_skus(args.skus)
    log.info("%d SKU unicos cargados", len(skus))
    con = init_db()
    ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if not args.solo_cruce:
        filtro = {s.strip().replace("www.", "") for s in args.sitios.split(",") if s.strip()}
        c = Cliente(args.pausa)
        for base in TIENDAS:
            dom = urlparse(base).netloc.replace("www.", "")
            if filtro and dom not in filtro:
                continue
            log.info("== %s", dom)
            try:
                filas, estado, metodo, nota = obtener_catalogo(c, base, skus, args)
            except Exception as e:  # un sitio con problemas no debe detener a los demas
                log.exception("%s: fallo inesperado", dom)
                filas, estado, metodo, nota = [], "ERROR", "-", str(e)[:200]
            guardar_catalogo(con, dom, filas, estado, metodo, nota, ahora)
            log.info("%s -> %s (%d productos, %s)", dom, estado, len(filas), metodo)

    cat = pd.read_sql("SELECT * FROM catalogo", con)
    estados = pd.read_sql("SELECT * FROM estados ORDER BY dominio", con)
    res = cruzar(skus, cat) if len(cat) else pd.DataFrame(columns=[
        "SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior", "Disponibilidad",
        "URL", "Coincidencia", "Fecha_catalogo", "Observaciones"])

    if len(res):
        h = res[["SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior",
                 "Disponibilidad", "URL", "Coincidencia"]].copy()
        h.insert(0, "ejecucion", ahora)
        h.columns = ["ejecucion", "sku", "competidor", "producto", "marca", "precio",
                     "precio_anterior", "disponibilidad", "url", "coincidencia"]
        h.to_sql("historial", con, if_exists="append", index=False)
    hist = pd.read_sql("SELECT * FROM historial ORDER BY sku, competidor, ejecucion", con)

    ruta = REPORTS_DIR / f"comparacion_precios_{datetime.now():%Y%m%d_%H%M}.xlsx"
    exportar(res, skus, estados, hist, ruta)
    res.to_csv(REPORTS_DIR / f"resultados_{datetime.now():%Y%m%d_%H%M}.csv", index=False,
               encoding="utf-8-sig")
    log.info("Listo. Exactas: %d | Probables: %d",
             (res["Coincidencia"] == "EXACTA").sum(), (res["Coincidencia"] == "PROBABLE").sum())


if __name__ == "__main__":
    main()

In [3]:
#!/usr/bin/env python3
"""
NEXOR - Comparador de precios GMD (v2)

Estrategia (para no saturar los sitios ni recibir HTTP 429):
  1. Descarga el CATALOGO de cada tienda en pocas peticiones:
       a) Shopify      -> /products.json          (250 productos por peticion)
       b) WooCommerce  -> /wp-json/wc/store/v1/products (100 por peticion)
       c) Respaldo     -> sitemap + JSON-LD de las paginas de producto (limitado)
  2. Guarda todo en SQLite (Datos/comparador.db).
  3. Cruza LOCALMENTE los SKU contra el catalogo descargado.
  4. Exporta Excel con 6 hojas y conserva historial entre ejecuciones.

Reglas de respeto:
  - Obedece robots.txt. Si el sitio lo restringe -> ACCESO_RESTRINGIDO, no se consulta.
  - Pausa entre peticiones (por dominio) y respeta Retry-After.
  - Ante HTTP 429 prolongado detiene ese dominio y lo registra (LIMITADO_429).
  - No accede a portales con login ni intenta evadir restricciones.

Uso:
  python comparador_precios.py                      # todo
  python comparador_precios.py --sitios saluti.com.co,medicolsa.com.co
  python comparador_precios.py --solo-cruce         # reusa catalogos ya descargados
  python comparador_precios.py --sin-fallback       # solo Shopify/WooCommerce
"""
import argparse
import html
import json
import logging
import random
import re
import sqlite3
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ----------------------------------------------------------------- configuracion
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:  # Colab/Jupyter: usa la carpeta de trabajo actual
    BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Datos"
REPORTS_DIR = BASE_DIR / "Reports"
LOGS_DIR = BASE_DIR / "Logs"
DB_PATH = DATA_DIR / "comparador.db"

UA = "NEXOR-PriceBot/2.0 (comparador interno de precios; contacto: ventas@nexor.co)"
TIMEOUT = (10, 30)   # (conexion, lectura) en segundos
PAUSA = 3.0              # segundos entre peticiones al mismo dominio
MAX_ESPERA_429 = 120     # si Retry-After supera esto, se detiene el dominio
MAX_PAGINAS_API = 200    # tope de paginas de catalogo por sitio
MAX_PAGINAS_FALLBACK = 150

TIENDAS = [
    "https://tiendamedicalospinos.com", "https://tmvco.com.co", "https://saluti.com.co",
    "https://ortopedicosfuturo.com", "https://gmd.com.co", "https://allers.com.co",
    "https://tecnomedica.com.co", "https://medicolsa.com.co", "https://novamedicasas.com",
    "https://imecolombia.com", "https://coimpormedica.com", "https://distrimedics.com",
    "https://promedco.com", "https://cimexcolombiasas.com", "https://tiendamedica.com.co",
    "https://tiendasmedicascolombia.com", "https://caldasmedicas.com",
    "https://corazonmedico.com.co", "https://dismecol.com", "https://tmmedicas.com",
    "https://labotekcolombia.com", "https://sidsalud.com", "https://medicox.com.co",
    "https://intercomercialmedica.com", "https://ortopedicos.com.co",
    "https://morthopedics.com", "https://medical-colombia.com", "https://bbraun.com",
    "https://sumimedical.com", "https://biosmedic.com", "https://jelt.com.co",
    "https://todomedico.co", "https://supermercadodelasalud.co",
    "https://santeyconfort.com", "https://insumedicosdelvalle.com",
    "https://medzoneortopedicos.com",
]

# Sitios que NO se deben consultar automaticamente (portal de distribuidores con login)
EXCLUIDOS = {"novamedicasas.com": "Precios solo en portal de distribuidores autorizados"}

log = logging.getLogger("nexor")


# ----------------------------------------------------------------- utilidades
def norm(s):
    return re.sub(r"[^A-Z0-9]", "", str(s or "").upper())


def to_float(v, api=False):
    if v is None or v == "":
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if api:
        try:
            return float(v)
        except ValueError:
            pass
    s = re.sub(r"[^\d.,]", "", str(v))
    if not s:
        return None
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", s):      # 1.234.567,89
        s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", s):    # 1,234,567.89
        s = s.replace(",", "")
    elif "," in s and "." not in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def leer_skus(ruta):
    skus, vistos = [], set()
    for linea in Path(ruta).read_text(encoding="utf-8", errors="ignore").splitlines():
        s = linea.strip()
        if not s or s.upper() == "N/D" or " " in s or "…" in s:
            continue
        if s not in vistos:
            vistos.add(s)
            skus.append(s)
    return skus


# ----------------------------------------------------------------- cliente HTTP respetuoso
class Cliente:
    def __init__(self, pausa=PAUSA):
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": UA, "Accept-Language": "es-CO,es;q=0.9"})
        self.pausa = pausa
        self.robots = {}
        self.bloqueado = {}   # base -> estado
        self.detalle = {}
        self.ultimo = {}

    def robots_de(self, base):
        if base in self.robots:
            return self.robots[base]
        rp = RobotFileParser()
        try:
            r = self.s.get(base + "/robots.txt", timeout=TIMEOUT)
            if r.status_code == 200:
                rp.parse(r.text.splitlines())
            elif 400 <= r.status_code < 500:
                rp.parse([])
            else:
                rp.parse(["User-agent: *", "Disallow: /"])
        except requests.ConnectionError as e:
            self.bloqueado[base] = "SITIO_NO_ACCESIBLE"
            self.detalle[base] = str(e).split("(Caused by")[-1][:150]
            rp.parse([])
        except requests.RequestException:
            rp.parse([])
        self.robots[base] = rp
        return rp

    def _esperar(self, base):
        falta = self.pausa + random.uniform(0, 1.0) - (time.time() - self.ultimo.get(base, 0))
        if falta > 0:
            time.sleep(falta)
        self.ultimo[base] = time.time()

    def get(self, base, url, params=None):
        """Devuelve (respuesta|None, motivo|None)."""
        if base in self.bloqueado:
            return None, self.bloqueado[base]
        if not self.robots_de(base).can_fetch(UA, url):
            return None, "ROBOTS"
        for intento in range(3):
            self._esperar(base)
            try:
                r = self.s.get(url, params=params, timeout=TIMEOUT)
            except requests.ConnectionError as e:
                self.bloqueado[base] = "SITIO_NO_ACCESIBLE"
                self.detalle[base] = str(e).split("(Caused by")[-1][:150]
                log.warning("%s: sin conexion (%s)", base, self.detalle[base])
                return None, "SITIO_NO_ACCESIBLE"
            except requests.RequestException as e:
                log.warning("%s: error de red %s", url, str(e)[:150])
                return None, "ERROR_RED"
            if r.status_code == 429:
                ra = r.headers.get("Retry-After", "")
                espera = int(ra) if ra.isdigit() else 30 * (intento + 1)
                log.warning("%s: HTTP 429 (espera %ss)", base, espera)
                if espera > MAX_ESPERA_429 or intento == 2:
                    self.bloqueado[base] = "LIMITADO_429"
                    log.error("%s: dominio detenido por HTTP 429", base)
                    return None, "LIMITADO_429"
                time.sleep(espera)
                continue
            if r.status_code >= 400:
                return None, f"HTTP_{r.status_code}"
            return r, None
        return None, "LIMITADO_429"


# ----------------------------------------------------------------- recolectores de catalogo
def catalogo_shopify(c, base):
    filas, pagina = [], 1
    while pagina <= MAX_PAGINAS_API:
        r, motivo = c.get(base, f"{base}/products.json", params={"limit": 250, "page": pagina})
        if r is None:
            return (filas or None), motivo
        try:
            prods = r.json().get("products")
        except (ValueError, AttributeError):
            return None, "NO_API"
        if prods is None:
            return None, "NO_API"
        if not prods:
            break
        for p in prods:
            for v in p.get("variants", []):
                var = v.get("title")
                titulo = p.get("title", "")
                if var and var != "Default Title":
                    titulo += f" - {var}"
                filas.append(dict(
                    sku=(v.get("sku") or "").strip(), titulo=titulo, marca=p.get("vendor", ""),
                    precio=to_float(v.get("price"), api=True),
                    precio_anterior=to_float(v.get("compare_at_price"), api=True),
                    disponibilidad="Disponible" if v.get("available") else "Agotado",
                    url=f"{base}/products/{p.get('handle')}", fuente="shopify"))
        pagina += 1
    return filas, None


def catalogo_woo(c, base):
    filas, pagina = [], 1
    while pagina <= MAX_PAGINAS_API:
        r, motivo = c.get(base, f"{base}/wp-json/wc/store/v1/products",
                          params={"per_page": 100, "page": pagina})
        if r is None:
            return (filas or None), motivo
        try:
            items = r.json()
        except ValueError:
            return None, "NO_API"
        if not isinstance(items, list):
            return None, "NO_API"
        if not items:
            break
        for it in items:
            pr = it.get("prices") or {}
            dec = 10 ** int(pr.get("currency_minor_unit", 0) or 0)
            precio = to_float(pr.get("price"), api=True)
            previo = to_float(pr.get("regular_price"), api=True)
            marcas = it.get("brands") or []
            filas.append(dict(
                sku=(it.get("sku") or "").strip(), titulo=html.unescape(it.get("name", "")),
                marca=", ".join(m.get("name", "") for m in marcas if isinstance(m, dict)),
                precio=precio / dec if precio is not None else None,
                precio_anterior=(previo / dec if previo is not None and previo != precio else None),
                disponibilidad="Disponible" if it.get("is_in_stock") else "Agotado",
                url=it.get("permalink", ""), fuente="woocommerce"))
        if len(items) < 100:
            break
        pagina += 1
    return filas, None


def _sitemaps(c, base, profundidad=0, urls_xml=None):
    if urls_xml is None:
        urls_xml = list(c.robots_de(base).site_maps() or []) or [
            f"{base}/sitemap.xml", f"{base}/sitemap_index.xml", f"{base}/wp-sitemap.xml"]
    paginas = []
    urls_xml = sorted(urls_xml, key=lambda u: 0 if "product" in u.lower() else 1)[:40]
    for u in urls_xml:
        r, _ = c.get(base, u)
        if r is None:
            continue
        locs = re.findall(r"<loc>\s*(.*?)\s*</loc>", r.text, flags=re.S)
        anidados = [x for x in locs if x.lower().endswith(".xml")]
        if anidados and profundidad < 2:
            paginas += _sitemaps(c, base, profundidad + 1, anidados)
        paginas += [x for x in locs if not x.lower().endswith(".xml")]
        if paginas:
            break
    return paginas


def _jsonld_producto(soup):
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except (ValueError, TypeError):
            continue
        pila = data if isinstance(data, list) else [data]
        while pila:
            d = pila.pop()
            if isinstance(d, dict):
                if "@graph" in d:
                    pila += d["@graph"]
                tipo = d.get("@type")
                tipos = tipo if isinstance(tipo, list) else [tipo]
                if "Product" in tipos:
                    return d
    return None


def catalogo_fallback(c, base, skus, max_paginas):
    todas = _sitemaps(c, base)
    patron = re.compile(r"/(product|producto|productos|products|shop|tienda|item)s?/", re.I)
    prod = [u for u in dict.fromkeys(todas) if patron.search(urlparse(u).path)]
    if not prod:
        return None, "SIN_SITEMAP_PRODUCTOS"
    claves = [s.lower() for s in skus if len(s) >= 5]
    prod.sort(key=lambda u: 0 if any(k in u.lower() for k in claves) else 1)
    filas = []
    for u in prod[:max_paginas]:
        r, motivo = c.get(base, u)
        if r is None:
            if motivo == "LIMITADO_429":
                break
            continue
        p = _jsonld_producto(BeautifulSoup(r.text, "html.parser"))
        if not p:
            continue
        of = p.get("offers") or {}
        of = of[0] if isinstance(of, list) and of else of
        of = of if isinstance(of, dict) else {}
        disp = str(of.get("availability", ""))
        marca = p.get("brand")
        marca = marca.get("name", "") if isinstance(marca, dict) else (marca or "")
        filas.append(dict(
            sku=str(p.get("sku") or p.get("mpn") or "").strip(),
            titulo=html.unescape(str(p.get("name", ""))), marca=marca,
            precio=to_float(of.get("price") or of.get("lowPrice")), precio_anterior=None,
            disponibilidad=("Disponible" if "InStock" in disp else "Agotado" if "OutOfStock" in disp
                            else "Sin informacion"),
            url=u, fuente="jsonld"))
    return (filas or None), None


def obtener_catalogo(c, base, skus, args):
    dom = urlparse(base).netloc.replace("www.", "")
    if dom in EXCLUIDOS:
        return [], "ACCESO_RESTRINGIDO", "-", EXCLUIDOS[dom]
    rp = c.robots_de(base)
    if c.bloqueado.get(base) == "SITIO_NO_ACCESIBLE":
        return [], "SITIO_NO_ACCESIBLE", "-", "No resuelve o no responde: " + c.detalle.get(base, "")
    if not rp.can_fetch(UA, base + "/"):
        return [], "ACCESO_RESTRINGIDO", "-", "robots.txt no permite el acceso automatizado"
    motivos = []
    for nombre, fn in (("shopify", catalogo_shopify), ("woocommerce", catalogo_woo)):
        filas, motivo = fn(c, base)
        if filas:
            return filas, "OK", nombre, ""
        motivos.append(motivo)
        if base in c.bloqueado:
            est = c.bloqueado[base]
            return [], est, nombre, c.detalle.get(base, "El servidor limito las solicitudes (HTTP 429)")
    if not args.sin_fallback:
        filas, motivo = catalogo_fallback(c, base, skus, args.max_paginas)
        if filas:
            return filas, "OK_PARCIAL", "jsonld", f"Cobertura parcial (max {args.max_paginas} paginas)"
        motivos.append(motivo)
        if base in c.bloqueado:
            est = c.bloqueado[base]
            return [], est, "jsonld", c.detalle.get(base, "El servidor limito las solicitudes (HTTP 429)")
    if motivos and all(m == "ROBOTS" for m in motivos if m):
        return [], "ACCESO_RESTRINGIDO", "-", "robots.txt"
    return [], "CATALOGO_NO_DETECTADO", "-", "Requiere adaptador especifico: " + ", ".join(m for m in motivos if m)


# ----------------------------------------------------------------- base de datos
def init_db():
    DATA_DIR.mkdir(exist_ok=True)
    con = sqlite3.connect(DB_PATH)
    con.executescript("""
    CREATE TABLE IF NOT EXISTS catalogo(dominio TEXT, sku TEXT, titulo TEXT, marca TEXT, precio REAL,
        precio_anterior REAL, disponibilidad TEXT, url TEXT, fuente TEXT, fecha TEXT);
    CREATE TABLE IF NOT EXISTS estados(dominio TEXT PRIMARY KEY, estado TEXT, metodo TEXT,
        productos INTEGER, nota TEXT, fecha TEXT);
    CREATE TABLE IF NOT EXISTS historial(ejecucion TEXT, sku TEXT, competidor TEXT, producto TEXT,
        marca TEXT, precio REAL, precio_anterior REAL, disponibilidad TEXT, url TEXT, coincidencia TEXT);
    """)
    return con


def guardar_catalogo(con, dom, filas, estado, metodo, nota, ahora):
    con.execute("DELETE FROM catalogo WHERE dominio=?", (dom,))
    con.executemany(
        "INSERT INTO catalogo VALUES(?,?,?,?,?,?,?,?,?,?)",
        [(dom, f["sku"], f["titulo"], f["marca"], f["precio"], f["precio_anterior"],
          f["disponibilidad"], f["url"], f["fuente"], ahora) for f in filas])
    con.execute("INSERT OR REPLACE INTO estados VALUES(?,?,?,?,?,?)",
                (dom, estado, metodo, len(filas), nota, ahora))
    con.commit()


# ----------------------------------------------------------------- cruce de SKU
def cruzar(skus, cat):
    cat = cat.reset_index(drop=True)
    cat["sku_norm"] = cat["sku"].fillna("").map(norm)
    textos = (cat["sku"].fillna("") + " | " + cat["titulo"].fillna("") + " | "
              + cat["url"].fillna("")).str.upper().tolist()
    idx = cat.groupby("sku_norm").indices
    filas = []

    def reg(sku, i, tipo):
        r = cat.iloc[i]
        obs = ("Verificar que presentacion/unidad de venta sea la misma" if tipo == "EXACTA"
               else "El SKU aparece en titulo/URL, no en el campo SKU: validar manualmente")
        filas.append(dict(SKU=sku, Competidor=r["dominio"], Producto=r["titulo"], Marca=r["marca"],
                          Precio=r["precio"], Precio_anterior=r["precio_anterior"],
                          Disponibilidad=r["disponibilidad"], URL=r["url"], Coincidencia=tipo,
                          Fecha_catalogo=r["fecha"], Observaciones=obs))

    for sku in skus:
        n = norm(sku)
        exactas = set(idx[n]) if len(n) >= 3 and n in idx else set()
        for i in exactas:
            reg(sku, i, "EXACTA")
        if len(n) >= 5:
            su = sku.upper()
            pat = re.compile(rf"(?<![A-Z0-9]){re.escape(su)}(?![A-Z0-9])")
            for i, t in enumerate(textos):
                if i not in exactas and su in t and pat.search(t):
                    reg(sku, i, "PROBABLE")
    cols = ["SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior", "Disponibilidad",
            "URL", "Coincidencia", "Fecha_catalogo", "Observaciones"]
    return pd.DataFrame(filas, columns=cols)


# ----------------------------------------------------------------- Excel
def exportar(res, skus, estados, hist, ruta):
    ex = res[res["Coincidencia"] == "EXACTA"].dropna(subset=["Precio"])
    if len(ex):
        g = ex.groupby("SKU")["Precio"]
        mejor = pd.DataFrame({"Precio_minimo": g.min(), "Precio_maximo": g.max(),
                              "Precio_promedio": g.mean().round(0), "Competidores": g.count()})
        imin = ex.loc[g.idxmin()].set_index("SKU")
        mejor["Competidor_mas_barato"] = imin["Competidor"]
        mejor["URL"] = imin["URL"]
        mejor = mejor.reset_index()
    else:
        mejor = pd.DataFrame(columns=["SKU", "Precio_minimo", "Precio_maximo", "Precio_promedio",
                                      "Competidores", "Competidor_mas_barato", "URL"])
    con_algo = set(res["SKU"])
    sin = pd.DataFrame({"SKU": [s for s in skus if s not in con_algo]})
    sin["Nota"] = "Sin coincidencia en los catalogos consultados (no implica que no exista en el mercado)"
    ok = estados[estados["estado"].isin(["OK", "OK_PARCIAL"])].shape[0]
    resumen = pd.DataFrame({"Indicador": [
        "SKU unicos consultados", "Sitios configurados", "Sitios con catalogo obtenido",
        "Sitios restringidos / limitados / sin catalogo", "SKU con coincidencia EXACTA",
        "SKU solo con coincidencia PROBABLE", "SKU sin coincidencia", "Fecha de ejecucion"],
        "Valor": [len(skus), len(estados), ok, len(estados) - ok,
                  res.loc[res["Coincidencia"] == "EXACTA", "SKU"].nunique(),
                  len(set(res.loc[res["Coincidencia"] == "PROBABLE", "SKU"])
                      - set(res.loc[res["Coincidencia"] == "EXACTA", "SKU"])),
                  len(sin), datetime.now().strftime("%Y-%m-%d %H:%M")]})
    estados = estados.rename(columns={"dominio": "Sitio", "estado": "Estado", "metodo": "Metodo",
                                      "productos": "Productos_en_catalogo", "nota": "Nota",
                                      "fecha": "Fecha"})
    with pd.ExcelWriter(ruta, engine="openpyxl") as w:
        for nombre, df in (("Resumen", resumen), ("Comparacion", res), ("Mejor precio", mejor),
                           ("Sin coincidencias", sin), ("Estado sitios", estados), ("Historial", hist)):
            df.to_excel(w, sheet_name=nombre, index=False)
            ws = w.sheets[nombre]
            ws.freeze_panes = "A2"
            for col in ws.columns:
                ancho = max(len(str(col[0].value or "")), *(len(str(x.value or "")) for x in col[:200]))
                ws.column_dimensions[col[0].column_letter].width = min(max(ancho + 2, 10), 60)
                if str(col[0].value).lower().startswith(("precio", "mejor")):
                    for x in col[1:]:
                        x.number_format = '"$"#,##0'
    log.info("Excel generado: %s", ruta)


# ----------------------------------------------------------------- principal
def main():
    ap = argparse.ArgumentParser(description="NEXOR - Comparador de precios GMD")
    ap.add_argument("--skus", default=str(BASE_DIR / "sku_gmd.txt"))
    ap.add_argument("--sitios", default="", help="dominios separados por coma (opcional)")
    ap.add_argument("--solo-cruce", action="store_true", help="no descarga; usa catalogos guardados")
    ap.add_argument("--sin-fallback", action="store_true", help="omite el respaldo sitemap+JSON-LD")
    ap.add_argument("--max-paginas", type=int, default=MAX_PAGINAS_FALLBACK)
    ap.add_argument("--pausa", type=float, default=PAUSA)
    args, _ = ap.parse_known_args()

    for d in (DATA_DIR, REPORTS_DIR, LOGS_DIR):
        d.mkdir(exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                        handlers=[logging.FileHandler(LOGS_DIR / "ejecucion.log", encoding="utf-8"),
                                  logging.StreamHandler()])
    skus = leer_skus(args.skus)
    log.info("%d SKU unicos cargados", len(skus))
    con = init_db()
    ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if not args.solo_cruce:
        filtro = {s.strip().replace("www.", "") for s in args.sitios.split(",") if s.strip()}
        c = Cliente(args.pausa)
        for base in TIENDAS:
            dom = urlparse(base).netloc.replace("www.", "")
            if filtro and dom not in filtro:
                continue
            log.info("== %s", dom)
            try:
                filas, estado, metodo, nota = obtener_catalogo(c, base, skus, args)
            except Exception as e:  # un sitio con problemas no debe detener a los demas
                log.exception("%s: fallo inesperado", dom)
                filas, estado, metodo, nota = [], "ERROR", "-", str(e)[:200]
            guardar_catalogo(con, dom, filas, estado, metodo, nota, ahora)
            log.info("%s -> %s (%d productos, %s)", dom, estado, len(filas), metodo)

    cat = pd.read_sql("SELECT * FROM catalogo", con)
    estados = pd.read_sql("SELECT * FROM estados ORDER BY dominio", con)
    res = cruzar(skus, cat) if len(cat) else pd.DataFrame(columns=[
        "SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior", "Disponibilidad",
        "URL", "Coincidencia", "Fecha_catalogo", "Observaciones"])

    if len(res):
        h = res[["SKU", "Competidor", "Producto", "Marca", "Precio", "Precio_anterior",
                 "Disponibilidad", "URL", "Coincidencia"]].copy()
        h.insert(0, "ejecucion", ahora)
        h.columns = ["ejecucion", "sku", "competidor", "producto", "marca", "precio",
                     "precio_anterior", "disponibilidad", "url", "coincidencia"]
        h.to_sql("historial", con, if_exists="append", index=False)
    hist = pd.read_sql("SELECT * FROM historial ORDER BY sku, competidor, ejecucion", con)

    ruta = REPORTS_DIR / f"comparacion_precios_{datetime.now():%Y%m%d_%H%M}.xlsx"
    exportar(res, skus, estados, hist, ruta)
    res.to_csv(REPORTS_DIR / f"resultados_{datetime.now():%Y%m%d_%H%M}.csv", index=False,
               encoding="utf-8-sig")
    log.info("Listo. Exactas: %d | Probables: %d",
             (res["Coincidencia"] == "EXACTA").sum(), (res["Coincidencia"] == "PROBABLE").sum())


if __name__ == "__main__":
    main()

In [5]:
import glob
import os
from google.colab import files

# Buscar todos los archivos de Excel generados en la carpeta reports
archivos = glob.glob('Reports/comparacion_precios_*.xlsx')

if archivos:
    # Obtener el archivo más reciente
    archivo_reciente = max(archivos, key=os.path.getctime)
    print(f"Descargando el archivo más reciente: {archivo_reciente}")
    files.download(archivo_reciente)
else:
    print("No se encontró ningún reporte en la carpeta 'Reports/'. Asegúrate de que el script principal ya se haya ejecutado exitosamente.")

Descargando el archivo más reciente: Reports/comparacion_precios_20261104_2234.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>